# Dioptra-DINO: End-to-End Architectural Ablation Retraining
### Retraining Component Controls from Scratch (Epoch 0 to 40) on GPU Acceleration

This autonomous kernel retrains the primary architectural ablations of Dioptra-DINO:
1. **`no-ara`**: Without Angular Residual Attention (`enable_ara=False`)
2. **`center-ray`**: Center-Ray PE Only (`ray_mode="center_ray"`, 36 dims vs 108 dims)
3. **`no-ray`**: Canonical 2D ViT-S/14 + DPT Decoder (`enable_trivision=False`)
4. **`no-vnl`**: Without 3D Virtual Normal Loss (`weight_normal=0.0`)


In [ ]:
# [1] Unpack Embedded Code & Configure GPU Hardware (T4 / P100 Compatibility)
import os, sys, io, base64, zipfile

print('=' * 75)
print('DIOPTRA-DINO ABLATION RUNTIME SETUP')
print('=' * 75)

# Check GPU architecture capability via nvidia-smi BEFORE importing torch
gpu_info = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null').read().strip()
print(f'NVIDIA GPU detected: {gpu_info}')
if 'P100' in gpu_info:
    print('Pascal GPU (P100 / sm_60) detected. PyTorch 2.6 dropped sm_60 support.')
    print('Installing PyTorch 2.4.1+cu121 for native sm_60 binary execution BEFORE importing torch...')
    os.system('pip install -q --no-cache-dir torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121')
    print('PyTorch 2.4.1 compatibility installation complete!')

# NOW import torch safely into Python
import torch
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    for i in range(gpu_count):
        major, minor = torch.cuda.get_device_capability(i)
        print(f'GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB VRAM, sm_{major}{minor})')
    print(f'Total available GPU acceleration: {gpu_count}x GPU(s) ready via PyTorch AMP.')
else:
    print('WARNING: CUDA is not available. Please verify GPU is enabled in notebook settings.')

# Unpack verified codebase
payload = '...'
buf = io.BytesIO(base64.b64decode(payload.encode('utf-8')))
with zipfile.ZipFile(buf, 'r') as zf:
    zf.extractall('/kaggle/working')
print('Extracted dioptra_dino.py and scripts/ successfully to /kaggle/working/')

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')


In [ ]:
# [2] Verify TartanAir Warehouse Stereo Suite Dataset
from dioptra_dino import resolve_dataset_root
data_root = resolve_dataset_root('auto')
print(f'Active TartanAir Dataset Root: {data_root}')


In [ ]:
# [3] Run Smoke Test Across All Ablation Configs (Pure Python Execution)
import torch, argparse
from scripts.train_dino_ablation import setup_ablation_config
from dioptra_dino import DioptraDINO, DioptraDINOLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Testing ablation models on device: {device}')

for ab in ['no-ara', 'center-ray', 'no-ray', 'no-vnl']:
    args = argparse.Namespace(epochs=40, batch_size=2, accum_steps=1, lr_backbone=2e-5, lr_head=2e-4, image_size=224)
    cfg = setup_ablation_config(ab, args)
    model = DioptraDINO(cfg).to(device)
    loss_fn = DioptraDINOLoss(cfg).to(device)
    x = torch.randn(2, 3, 224, 224, device=device)
    d = torch.abs(torch.randn(2, 1, 224, 224, device=device)) + 1.0
    K = torch.eye(3, device=device).unsqueeze(0).expand(2, -1, -1)
    pred = model(x, K)
    loss, _ = loss_fn(pred, d, K=K)
    loss.backward()
    print(f'Verification [{ab:12s}] PASS! Params: {sum(p.numel() for p in model.parameters())/1e6:.3f}M')


--- 
## Execute Retraining of Ablation Variants


In [ ]:
# [Ablation 1 Status] 'no-ara' was previously trained for 40 epochs on Kaggle,
# verified, and benchmarked on 200 held-out frames (AbsRel: 0.5516, Scale: 0.4901).
print('[Ablation 1] no-ara already completed and verified.')


In [ ]:
# [Ablation 2] Retrain CENTER-RAY ONLY PE (ray_mode='center_ray')
# Evaluates single optical direction vs full Trivision frustum aperture
!python scripts/train_dino_ablation.py --ablation center-ray --epochs 20 --batch-size 16 --accum-steps 2

# Evaluate immediately on 200 held-out frames
!python scripts/download_and_eval_200.py --checkpoint 'outputs_ablations/ablation_center_ray/dioptra_dino_center_ray_best.pt' --output-dir 'outputs_ablations/ablation_center_ray/eval_200' --save-json 'outputs_ablations/ablation_center_ray/eval_200_metrics.json'
!zip -r -q dioptra_dino_ablations_retrained.zip outputs_ablations/
print('Ablation 2 [center-ray] completed, evaluated, and checkpoint archived.')


In [ ]:
# [Ablation 3] Retrain WITHOUT Ray Positional Modulation (Canonical 2D ViT + DPT)
# Evaluates whether foundation features alone can establish calibrated metric scale
!python scripts/train_dino_ablation.py --ablation no-ray --epochs 20 --batch-size 16 --accum-steps 2

# Evaluate immediately on 200 held-out frames
!python scripts/download_and_eval_200.py --checkpoint 'outputs_ablations/ablation_no_ray/dioptra_dino_no_ray_best.pt' --output-dir 'outputs_ablations/ablation_no_ray/eval_200' --save-json 'outputs_ablations/ablation_no_ray/eval_200_metrics.json'
!zip -r -q dioptra_dino_ablations_retrained.zip outputs_ablations/
print('Ablation 3 [no-ray] completed, evaluated, and checkpoint archived.')


In [ ]:
# [Ablation 4] Retrain WITHOUT 3D Virtual Normal Loss (weight_normal=0.0)
# Evaluates planar architectural floor/wall consistency without normal supervision
!python scripts/train_dino_ablation.py --ablation no-vnl --epochs 20 --batch-size 16 --accum-steps 2

# Evaluate immediately on 200 held-out frames
!python scripts/download_and_eval_200.py --checkpoint 'outputs_ablations/ablation_no_vnl/dioptra_dino_no_vnl_best.pt' --output-dir 'outputs_ablations/ablation_no_vnl/eval_200' --save-json 'outputs_ablations/ablation_no_vnl/eval_200_metrics.json'
!zip -r -q dioptra_dino_ablations_retrained.zip outputs_ablations/
print('Ablation 4 [no-vnl] completed, evaluated, and checkpoint archived.')


--- 
## Final Checkpoint Verification & Archive Packaging


In [ ]:
# [Summary] List all saved checkpoints and final archive
import os
print('\nFinal list of saved ablation checkpoints:')
os.system('ls -lh outputs_ablations/*/*.pt')
os.system('zip -r -q dioptra_dino_ablations_retrained.zip outputs_ablations/')
print('Final archive ready: /kaggle/working/dioptra_dino_ablations_retrained.zip')
